# Regression: Price Prediction
Predict car price; target is log-transformed to reduce skew and RMSE.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('regression_features.csv')
X, y = df.drop(columns=['price']), df['price']
y_log = np.log1p(y)

X_train, X_test, y_train_log, y_test_log = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)
y_test_orig = np.expm1(y_test_log)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train_log, test_size=0.1, random_state=42
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (597524, 7), Test: (149382, 7)


In [3]:
results = {}

def evaluate(name, y_pred_log):
    y_pred = np.expm1(np.clip(y_pred_log, 0, 15))
    rmse = np.sqrt(mean_squared_error(y_test_orig, y_pred))
    mape = mean_absolute_percentage_error(y_test_orig, y_pred) * 100
    results[name] = {'RMSE': round(rmse, 2), 'MAPE%': round(mape, 2)}
    print(f'{name:<35} RMSE: ${rmse:>10,.0f}   MAPE: {mape:.2f}%')

In [4]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

ridge = Ridge(alpha=10.0)
ridge.fit(X_train_sc, y_train_log)
evaluate('Ridge Regression', ridge.predict(X_test_sc))

Ridge Regression                    RMSE: $    10,874   MAPE: 23.45%


In [5]:
rf = RandomForestRegressor(
    n_estimators=200, max_depth=25, min_samples_leaf=3,
    max_features='sqrt', n_jobs=-1, random_state=42
)
rf.fit(X_train, y_train_log)
evaluate('Random Forest', rf.predict(X_test))

Random Forest                       RMSE: $     4,577   MAPE: 9.41%


In [6]:
xgb_model = xgb.XGBRegressor(
    n_estimators=3000, learning_rate=0.03, max_depth=7,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0, min_child_weight=5,
    random_state=42, n_jobs=-1, verbosity=0,
    early_stopping_rounds=50, eval_metric='rmse'
)
xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
evaluate('XGBoost', xgb_model.predict(X_test))
print(f'  Best iteration: {xgb_model.best_iteration}')

XGBoost                             RMSE: $     4,627   MAPE: 9.49%
  Best iteration: 2999


In [7]:
lgb_model = lgb.LGBMRegressor(
    n_estimators=3000, learning_rate=0.03, max_depth=8,
    num_leaves=127, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0, min_child_samples=20,
    random_state=42, n_jobs=-1, verbosity=-1
)
lgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
)
evaluate('LightGBM', lgb_model.predict(X_test))
print(f'  Best iteration: {lgb_model.best_iteration_}')

LightGBM                            RMSE: $     4,467   MAPE: 9.23%
  Best iteration: 3000


In [8]:
et = ExtraTreesRegressor(
    n_estimators=200, max_depth=25, min_samples_leaf=3,
    max_features='sqrt', n_jobs=-1, random_state=42
)
et.fit(X_train, y_train_log)
evaluate('Extra Trees', et.predict(X_test))

Extra Trees                         RMSE: $     5,780   MAPE: 11.14%


In [9]:
results_df = pd.DataFrame(results).T.sort_values('RMSE')
print(results_df.to_string())
print(f'\nBest model: {results_df.index[0]}')

                      RMSE  MAPE%
LightGBM           4466.99   9.23
Random Forest      4577.20   9.41
XGBoost            4627.42   9.49
Extra Trees        5779.79  11.14
Ridge Regression  10873.77  23.45

Best model: LightGBM
